# GortsUp Academy
## Ֆայլերի գործնական մշակում իրական տվյալներով
**Data Analytics · Առկա դաս · Ռաֆիկ Առաքելյան**

Լուծեք 15 վարժությունները հերթով։ Յուրաքանչյուր վարժությունում կա պայման, ձեր լուծման բջիջ, առաջարկվող լուծում և ակտիվ assert ստուգումների բջիջ։ Ձեր տարբերակը գրելուց հետո բաց թողեք առաջարկվող լուծումը և գործարկեք ստուգումները։ Ամբողջ notebook-ը հերթով գործարկելիս ստուգվում են առաջարկվող լուծումները։

Սկզբում կաշխատենք ուղիների ու տեքստային ֆայլերի վրա, ապա իրական CSV-ից կստանանք մաքրված տվյալներ և ամփոփիչ հաշվետվություն։ Փոքր սխալ մուտքերն ու մատյանի տեքստերը հատուկ ստեղծված ուսումնական օրինակներ են։

### Իրական տվյալաշարը
Օգտագործում ենք UCI Wine Quality-ի կարմիր գինու բոլոր 1,599 դիտարկումները։ Ներառված են 12 բնօրինակ սյունակներից երեքը՝ alcohol, pH և quality։ Արժեքներն ու հերթականությունը պահպանված են։ alcohol-ը ծավալային տոկոս է, pH-ը՝ թթվայնության ցուցանիշ, quality-ն՝ փորձագետների գնահատական։

Հեղինակներ՝ P. Cortez, A. Cerdeira, F. Almeida, T. Matos և J. Reis։ [Տվյալաշարի պաշտոնական էջ](https://doi.org/10.24432/C56S3T)

Տվյալները ներկառուցված են notebook-ում։ Լրացուցիչ փաթեթներ կամ ներբեռնում պետք չեն։


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory, NamedTemporaryFile
from math import isfinite
import csv
import json 
import os

In [3]:
DATA_CSV = '''alcohol,pH,quality
9.4,3.51,5
9.8,3.2,5
9.8,3.26,5
9.8,3.16,6
9.4,3.51,5
9.4,3.51,5
9.4,3.3,5
10,3.39,7
9.5,3.36,7
10.5,3.35,5
9.2,3.28,5
10.5,3.35,5
9.9,3.58,5
9.1,3.26,5
9.2,3.16,5
9.2,3.17,5
10.5,3.3,7
9.3,3.11,5
9,3.38,4
9.2,3.04,6
9.4,3.39,6
9.7,3.52,5
9.5,3.17,5
9.4,3.17,5
9.7,3.43,6
9.3,3.34,5
9.5,3.28,5
9.5,3.17,5
9.4,3.47,5
9.8,3.38,6
10.1,3.35,5
10.6,3.46,6
9.8,3.17,5
9.4,3.45,6
9.2,3.38,5
9.6,3.4,6
10.8,3.42,6
9.7,3.23,7
9.8,3.5,4
10.5,3.33,5
10.5,3.33,5
9.3,3.26,4
10.5,3.21,6
10.3,3.3,5
9.5,3.48,5
13.1,3.9,4
9.2,3.25,5
9.5,3.25,5
9.2,3.34,5
9.2,3.32,5
9.2,3.15,5
9.4,3.4,6
9.4,3.39,6
9.4,3.2,5
10.2,3.17,6
9.5,3.34,5
9.6,3.04,5
9.4,3.26,5
10,3.43,5
9.4,3.41,6
9.2,3.44,5
9.3,3.21,5
9.5,3.38,7
9.8,3.39,5
10.9,3.41,5
10.9,3.41,5
9.6,3.42,5
10.7,3.44,5
10.7,3.28,5
10.5,3.34,6
9.5,3.32,6
9.5,3.32,5
9.5,3.31,5
9.2,3.31,4
9.6,3.28,5
10.5,3.54,5
10.5,3.54,5
10.7,3.52,6
10.1,3.38,5
9.1,3.16,4
9.2,3.41,5
9.4,3.13,5
9.1,3.36,5
9.4,3.16,5
10.3,3.44,6
10.1,3.41,5
9.9,2.93,6
9.6,3.39,5
9.5,3.14,5
9,3.48,5
9.5,3.23,5
9.9,2.93,6
9.8,2.93,5
9.6,3.39,5
10.5,3.75,4
12.9,3.85,6
10.7,3.45,5
9.2,3.25,5
9.8,3.39,5
9,3.3,6
10.2,3.4,6
10.4,3.31,6
9,3.3,6
9.2,3.29,5
9.4,3.33,5
9.2,3.29,5
9.3,3.08,5
9.3,3.46,5
9.6,3.37,6
9.3,3.21,5
9.5,3.19,5
9.8,3.15,5
9.8,3.15,5
9.7,3.32,6
9.5,3.19,5
10.5,3.07,6
10,3.39,6
9.4,3.37,6
10.9,3.21,6
9.2,3.34,6
9,3.3,5
10.9,3.21,6
9.2,3.49,5
9.5,3.44,5
9.5,3.39,5
9.4,3.16,5
10.9,3.53,5
10.9,3.54,5
10.5,3.42,7
9.4,3.42,5
9.4,3.24,5
13,3.63,5
13,3.63,5
9.8,3.4,6
9.9,3.22,6
9.6,3.19,5
9.5,3.23,5
9.2,3.48,5
9.5,3.33,5
9.5,3.32,5
9.6,3.19,5
9.5,3.23,5
14,3.68,6
9.4,3.34,5
14,3.68,6
9.4,3.17,5
10,3.54,5
9.3,3.11,5
10.2,3.42,6
10.5,3.53,6
10.3,3.33,6
9.4,2.74,4
10.1,3.35,5
10.1,3.35,5
10.5,3.42,5
10.5,3.42,5
10.5,3.42,5
10.5,3.42,5
9.3,3.48,5
9.3,3.59,6
9.6,3.2,5
9.2,3.17,4
10,3.33,6
9.4,3.37,5
9.4,3.37,5
9.5,3.19,5
10.2,3.34,5
9,3.37,4
10.4,3.47,6
9.5,3,5
9.1,3.36,4
9.2,3.29,6
9.2,3.29,6
11.5,3.42,6
9.5,3.33,5
9.5,3.35,5
9.5,3.33,5
10.5,3.58,6
9.6,3.48,5
9.5,3.19,5
9.5,3.19,5
9.3,3.12,5
9.3,3.44,5
9.3,3.54,5
9.3,3.52,6
9.7,3.26,5
9.2,3.34,5
9.7,3.39,5
9.5,3.2,5
9.5,3.2,5
9.4,3.23,5
9.8,3.57,6
9.5,3.28,5
9.7,3.28,5
9.7,3.28,5
9.4,3.25,5
10.2,3.46,5
10.1,3.11,6
13,3.57,7
11.4,3.51,4
10.3,3.22,7
9.3,3.04,5
9.5,3.36,5
9.2,3.34,5
9.2,3.34,6
10.8,3.2,7
10.8,3.2,7
9.3,3.29,5
9.4,3.22,5
10.5,3.31,7
12.4,3.3,6
10,3.35,6
10.2,3.21,6
10.1,3.3,5
9.8,3.24,6
10.5,3.34,5
11,3.14,5
9.1,3.36,5
9.7,3.39,5
9.5,3.3,5
9.4,3.34,6
9.4,3.15,5
9.5,3.42,5
10,3.28,6
10.4,3.31,4
10.5,3.37,6
9.5,3.04,6
9.8,3.39,5
10.5,3.37,6
11,3.46,5
12.2,3.54,7
9.9,3.37,6
9.6,3.34,6
11,3.46,5
9,3.32,6
9,3.37,6
9,3.37,6
9.2,3.37,6
9,3.37,6
9,3.32,6
9.3,3,5
10.9,3.14,6
9.8,3.08,6
9.2,3.07,7
9.2,3.07,7
9.9,3.61,6
9.5,3.38,5
9.3,3.2,5
9.8,3.35,6
9.9,3.61,6
10,3.22,6
9.9,3.47,6
10.5,3.17,5
9.5,3.23,5
9.9,3.47,6
9.3,3.26,5
9.2,3.22,5
9.2,3.49,5
9.4,3.06,5
10.5,3.36,7
9.3,3.29,5
9.4,3.35,4
10,3.34,5
9.3,3.28,5
10.9,3.08,5
10.2,3.2,7
9.8,3.6,4
12.8,3.35,8
9.4,3.69,6
10.1,3.28,6
10.7,3.36,6
10.1,3.28,6
10.1,3.22,5
9.4,3.31,5
9.4,3.38,5
10.7,3.36,6
9.4,3.69,6
10.1,3.28,6
12.6,3.23,8
10.5,3.38,7
9.3,3.12,6
9.9,3.25,7
9.1,3.36,5
10.5,3.38,7
9.8,3.31,5
9.8,3.31,5
10.3,3.1,6
10.3,3.37,6
10.6,3.34,7
9.2,3.2,5
10.6,3.34,7
10.5,3.3,5
10.3,3.22,6
10.1,3.41,6
9.5,3.05,6
9.5,3.05,5
9.9,3.16,5
9.6,3.67,5
9.7,3.67,5
9.6,3.68,5
10.7,3.38,6
10.1,3.22,6
10,3.51,5
9.5,3.39,5
9.2,3.2,5
9.3,3.12,6
9.4,3.36,5
9.5,3.19,6
9.5,3.19,6
9.5,3.41,6
9.3,3.12,6
9.4,3.27,6
9.5,3.32,6
9.4,3.3,5
11,3.39,5
11,3.42,6
10.1,3.3,5
10.4,3.3,6
11.5,3.37,7
10.4,3.3,6
11.5,3.37,7
9.7,3.24,5
9.3,3.29,5
9.5,3.28,6
9.2,3.16,6
9.2,3.16,6
11.5,3.05,7
11.5,3.28,5
9.7,3.16,6
9.5,3.18,5
12.5,3.23,6
12.5,3.23,6
9.4,3.22,6
11,3.13,5
11.7,3.34,7
12.2,3.17,7
12.5,3.35,6
10.3,3.31,5
11.5,3.16,6
9.8,3.11,7
9.2,3.12,6
11.3,3.21,6
9.8,3.3,6
9.8,3.3,6
10.7,3.12,6
9.9,3.6,5
12.3,3.58,7
12,3.02,6
10,3.11,6
9.4,3.36,6
9.9,3.28,6
9.4,3.35,6
9.3,3.27,5
13,3.18,5
11.9,3.25,6
12.8,3.55,6
11,3.26,5
11.7,3.24,7
10.4,3.15,7
9.8,3.17,6
9.4,3.19,5
9.9,3.37,6
10,3.05,5
10.2,2.99,5
10,3.07,7
11.8,3.26,6
10,3.07,7
9,3.16,5
9.4,3.18,5
12,3.2,7
9.9,3.57,5
8.7,3.32,6
10.6,3.32,6
9.2,3.05,5
10.8,3.01,6
11.8,3.01,7
11,3.26,6
12,3.2,7
13.3,3.11,6
10.8,3.31,6
9.4,3.21,6
10,3.06,6
9.4,3.21,6
9.4,3.21,6
9.2,3.42,5
9.7,3.43,6
9.2,3.41,6
9.6,3.31,6
9.2,3.43,6
10,3.19,7
12.9,3.56,8
10,3.06,6
9.5,3.18,5
9.1,3.16,5
9.9,3.03,5
13,3.22,7
9.9,3.47,5
11,3.18,6
11,3.18,6
9.4,3.1,5
9.9,3.47,5
10.8,3.29,6
10.5,3.1,6
10.5,3.15,6
9.1,3.26,5
10.1,3.36,6
10.8,3.24,6
10.8,3.18,7
11.3,3.19,6
9.6,3.05,4
9.5,3.38,6
9.5,3.38,5
9.3,3.27,5
11.7,3.27,7
9.5,3.26,5
9.3,3.32,5
11.7,3.26,6
10.5,3.44,5
10.4,3.24,6
9.9,3.62,5
11.8,3.28,7
12.3,3.52,7
10.9,3.36,5
11,3.15,7
10.9,3.36,5
12.3,3.52,7
11.4,3.49,6
10.6,3.26,6
9.3,3.24,5
10.4,3.1,6
11,3.15,7
9.2,3.25,5
12.8,3,6
9.5,3.16,5
9.9,3.15,6
9.5,3.16,5
10.2,3.26,6
11.2,3.17,6
9.9,3.15,6
9.3,3.08,5
9.8,2.88,8
11.3,3.06,6
11.2,2.95,7
11.6,3.23,7
12.5,3.57,7
10.1,3.23,6
10.5,3.1,5
11.2,3.22,5
10.2,3.2,6
10.8,3.1,6
10.8,3.1,6
9.1,3.06,6
10,3.44,6
11.2,3.26,7
11.1,3.5,5
13.4,3.22,8
10.3,3.37,5
9.6,3.28,5
11.2,3.26,7
9,3.25,3
11.3,3.35,6
9.3,3.26,5
11.8,3.18,5
9,3.08,5
9.2,2.98,6
9.7,3.48,5
11.5,3.16,6
14,3.32,6
9.2,3.17,6
9.8,3.2,5
10.6,3.05,5
11.4,3.2,6
10.4,3.15,6
10.6,3.26,5
9.4,3.2,6
10.2,3.14,5
9.7,3.28,5
11,3.18,6
10.2,3.14,5
10.1,3.19,6
9.2,3.12,5
11.7,3.15,8
9.4,3.04,5
9.4,3.04,5
13.4,3.06,6
10,3.17,5
10,3.17,5
10,3.17,6
10.8,3.32,7
10.2,3.34,6
10.6,3.31,6
13.3,3.34,7
13.4,3.32,7
11.6,3.48,6
12.1,3.44,6
11,3.15,8
9,3.43,6
11.1,3.32,5
11,3.15,8
11.6,3.48,6
9,3.43,6
12,3.17,7
12,3.17,7
10.9,3.18,7
10.8,3.21,7
12.5,3.16,7
10.8,3.25,7
9.5,3.17,6
10.2,3.18,6
11.4,3.04,7
9.5,3.02,5
10.2,3.18,6
9.7,3.03,6
11.8,3.09,7
11.8,3.09,7
9.3,3.31,5
11.9,3.25,6
8.4,3.16,3
11.7,3.22,6
11,3.36,5
10,3.42,6
9.1,3.48,5
9.8,3.38,5
9.4,3.22,5
9.5,3.23,5
9.9,3.24,5
11,3.36,5
11.4,3.39,6
8.7,3.34,6
9.4,3.09,5
10.3,3.41,6
10.3,3.16,5
10.3,3.16,5
12.8,3.22,6
10,3.28,6
10.3,3.41,6
9.4,3.09,5
10.7,3.37,6
12,3.2,7
11.2,3.22,5
9.6,3.26,5
11,3.21,6
9.9,3.12,5
11,3.06,6
8.4,2.86,6
9.1,3.08,5
9.5,3.45,6
10.7,3.26,6
10.4,3.12,6
9.4,3.27,6
9.5,3.33,6
10,3.19,6
10,3.12,6
11.5,3.74,5
11.1,2.92,5
11.1,2.92,5
11.7,3.07,6
11.1,2.92,5
11.7,3.07,6
12.7,3.3,6
11.4,3.14,5
9.2,3.23,5
9.2,3.08,5
10.1,3.31,6
12.7,3.3,6
11.4,3.14,5
9,3.32,6
9,3.32,6
10.7,3.31,6
11.7,3.57,6
11,3.32,6
11.7,3.57,6
10.4,3.14,5
9.6,3.3,4
10,3.19,6
10.2,3.2,6
10,3.34,4
9.5,3.3,5
9.8,3.29,5
9.8,3.14,6
9.6,3.19,5
9.6,3.19,5
9.2,3.19,5
9.9,2.98,7
10.7,3.3,7
9.6,3.47,6
10.6,3.12,7
9.3,3.26,5
14,3.72,8
10.5,3.05,7
9.7,3.35,5
11.5,3.12,6
9.7,3.35,5
9,3.29,5
9.5,3.4,5
9.3,3.27,5
9.3,3.04,6
10,3.09,6
9.8,3.2,6
9.3,3,6
10,3.26,4
9,3.1,6
9.3,3.3,5
9,3.1,6
9.1,3.17,6
9.2,3.31,6
12.2,3.34,7
10.5,3.31,6
10.4,3.32,6
12.7,3.56,6
9.2,3.39,5
9.4,2.98,5
10,3.32,6
9.8,3.26,5
10.2,2.87,6
9.7,3.5,5
9.7,3.5,5
9.8,3.03,6
10.2,3.13,5
9.3,3.09,5
9.4,3.27,5
9.4,3.28,5
9.5,3.13,5
12.1,3.49,6
10.2,3.46,5
10.2,3.46,5
9.1,3.36,5
9.1,3.36,5
9.3,3.27,6
9.3,3.21,5
9.3,3.27,6
9.5,3.25,5
10.5,3.32,6
11.3,3.43,4
9.5,3.27,5
9.7,3.27,5
9.4,3.24,5
9.4,3.24,5
10.2,3.35,7
10.3,3.18,6
9.4,3.39,5
9.5,3.46,5
9.4,3.39,5
9.5,3.46,5
9.4,3.39,5
10.1,3.37,7
10.1,3.4,5
11,3.32,4
11.2,3.33,7
11.3,3.16,6
9.6,2.89,5
11.2,3.41,5
14.9,2.98,5
12,3.12,6
9.5,3.08,5
9.4,3.14,5
9.6,2.89,5
10.5,2.92,7
9.6,3.32,6
11,3.41,4
9.6,3.32,6
9,3.31,5
9.6,3.36,6
10.2,3.04,6
10.2,3.03,5
9.7,3.14,5
9.5,3.18,6
9.2,2.94,6
11,3.15,5
9.2,2.94,6
10,3.26,5
9.5,3.2,5
9.5,3.15,5
9.5,3.2,5
9.3,3.08,6
10.2,3.26,5
9.3,3.08,6
9.9,3.4,5
10,3.48,5
9.6,3.13,5
9,3.06,5
10.2,3.21,6
9.8,3.33,5
11.3,3.37,5
9.4,3.25,5
11.3,3.37,5
9.1,3.44,5
9.7,3.28,5
9.4,3.4,5
9.4,3.3,5
10.7,3.63,3
9.8,3.48,5
9,3.03,5
9.4,3.32,5
9.4,3.31,5
12.8,3.9,6
9.5,3.47,6
9.5,3.47,6
9.7,3.31,5
10.8,3.11,6
10.1,3.08,6
9.5,3.47,6
9.4,3.47,6
9.6,3.32,4
9.7,3.29,4
9.9,3.37,5
10,3.4,5
10.5,3.37,5
11.6,3.34,6
10,3.27,6
10.1,3.06,5
9.5,3.38,5
9.4,3.36,5
9.4,3.34,5
9.8,3.28,5
9.2,3.29,6
9.4,3.34,5
10,3.21,5
9.6,3.22,5
9.5,3.46,5
9.6,3.22,5
9.2,3.1,5
10,3.21,5
9.5,2.94,5
11.2,3.54,4
10.4,3.29,5
11.1,3.43,6
9.5,3.46,5
9.5,3.46,5
12.7,3.61,6
9.6,3.17,5
11.5,3.44,5
9.6,3.39,5
9.5,3.55,5
9.3,3.48,5
9.5,3.26,5
9.5,3.26,5
9.3,3.5,6
9.2,3.1,5
9.3,3.35,5
11.5,3.45,6
9.5,3.21,5
9.2,3.46,5
10,3.13,5
9.5,3.13,5
9.5,3.52,6
9,3.3,6
9.4,3.3,5
9.6,3.52,6
9.5,3.52,6
9.5,3.29,5
9.5,3.29,5
9.4,3.39,5
9.5,3.29,5
9.1,3.09,6
10.7,3.51,6
11.2,3.6,6
9.8,3.25,5
9.8,3.25,5
9.2,3.19,5
9.7,3.23,5
9.6,3.25,5
10,3.31,6
9.6,3.25,5
9.5,3.31,6
9.5,3.31,6
9.4,3.14,5
9.5,3.45,5
9.7,3.55,6
9.6,3.39,5
9.7,3.55,6
9.4,3.28,5
9.4,3.27,5
9.5,3.3,6
9.5,3.31,6
10,3.43,5
10.3,3.4,6
10.3,3.4,6
10.5,3.36,5
9.8,3.5,5
9.4,3.31,6
9.8,3.66,5
10,3.36,5
9.8,3.66,5
9.8,3.55,5
9.5,3.21,5
9.5,3.21,5
10.1,3.18,6
10.1,3.18,6
9.3,3.09,5
9.7,3.33,6
9.6,3.23,5
9.7,3.48,6
10.8,3.38,5
12.5,3.17,6
10.2,2.99,5
9.6,3.1,5
10.8,3.24,7
10.7,3.29,6
10.7,3.29,6
9.4,3.25,5
10,3.23,5
12.9,3.56,7
9.6,3.24,6
9.9,3.26,6
12.8,3.33,7
12.5,3.27,7
12.8,3.33,7
9.2,3.26,5
10.3,3.22,6
10.5,3.3,5
10.9,3.09,6
10.8,3.24,5
11.4,3.44,4
11.3,3.21,6
10.8,3.24,5
10.5,3.23,6
11.9,3.28,6
9.4,3.29,5
9.6,3.28,5
9.7,3.34,5
14,3.71,7
9.8,3.61,5
9.8,3.61,5
10.3,3.24,5
10.7,3.36,5
11,3.4,7
10.7,3.36,5
12.7,3.46,8
11.1,3.56,6
10.9,3.38,4
11.1,3.56,6
9.9,3.38,3
9.9,3.38,4
9.4,3.32,5
9.3,3.39,5
11.7,3.26,7
11.7,3.26,7
11.2,3.24,7
10,3.46,5
12.1,3.24,7
10.3,3.45,5
10.9,3.23,6
9.4,3.28,5
10.6,3.18,6
9.8,3.59,5
9.8,3.59,5
9.9,3.5,6
9.8,3.59,5
9.8,3.58,5
9.5,3.28,5
9.5,3.28,5
9.7,3.22,5
10.9,3.47,6
10.9,3.47,6
11.7,3.51,7
10.9,3.47,6
11.3,3.4,7
10.6,3.15,7
11.2,3.4,6
9.5,3.51,5
11.5,3.66,6
10.4,3.24,5
9.7,3.51,5
9.5,3.51,5
9.7,3.51,5
11.3,3.41,6
11.2,3.4,6
11.2,3.4,6
10.9,3.44,6
11.8,3.41,6
10.6,3.53,5
10,3.43,4
11.1,3.23,7
11.8,3.12,7
11.8,3.31,7
10.9,3.4,4
11.8,3.41,6
10,3.22,6
9.5,3.53,5
9.9,3.15,5
11.4,3.48,6
12,3.22,6
9.5,3.53,5
10,3.22,6
10.5,3.34,5
10.4,3.33,6
12.1,3.15,7
11.2,3.43,6
9.3,3.22,5
10.1,3.44,5
9.7,3.53,5
9.3,3.22,6
9.7,3.53,5
9.8,3.53,6
10.7,3.42,6
12.5,3.31,7
10.7,3.42,6
12.5,3.31,7
11,3.48,3
11.8,3.4,5
10.8,3.58,7
10.8,3.58,7
10.8,3.41,7
10.8,3.41,7
9.5,3.23,5
11,3.39,5
11.5,3.59,6
10.8,3.43,6
11.5,3.29,6
13.2,3.23,6
10.9,3.18,6
12.2,3.33,6
12.2,3.27,7
11.5,3.29,6
11.9,3.27,6
11,3.62,5
10.1,3.38,6
11,3.3,6
11.8,3.38,6
10.5,3.24,5
11,3.3,6
11.8,3.38,6
10.1,3.38,6
11.2,3.18,5
11,3.47,7
10.2,3.47,6
9.2,3.2,4
11.2,3.18,5
12,3.3,7
10.5,3.69,5
9.8,3.48,5
9.5,3.45,6
9.8,3.48,5
10.5,3.69,5
11.8,3.39,6
11.8,3.39,6
10.4,3.07,4
12.9,3.34,7
11.4,3.61,5
12.4,3.36,7
12.5,3.19,7
10,3.22,7
9.7,3.24,7
12.1,3.27,7
11.1,3.14,7
12,3.29,7
12.4,3.26,7
11.9,3.41,7
11.9,3.41,7
11.9,3.41,7
12.4,3.26,7
11.2,3.31,7
12.1,3.2,7
12.2,3.44,6
10.4,3.36,5
11.3,3.31,6
11.1,3.29,6
11.3,3.47,7
10,3.36,5
11.1,3.16,6
9.3,3.27,5
9.5,3.38,5
11.2,3.15,6
11.1,3.16,6
11,3.34,6
11.9,3.13,7
9.2,3.13,5
12.2,3.2,6
10.2,3.46,5
10.9,3.33,6
10.9,3.33,6
11.6,3.22,7
10.7,3.52,5
12.1,3.3,7
9.4,3.44,5
9.4,3.44,5
9.1,3.15,5
11.3,3.28,7
10.4,3.13,5
10.5,3.32,6
10,3.25,5
12.9,3.34,6
10.5,3.32,6
10.4,3.13,5
11.3,3.45,6
11.4,3.19,7
9.4,3.44,5
9.4,3.16,5
10.9,3.14,6
9.4,3.16,5
9.4,3.45,5
9.4,3.36,6
9.4,3.45,5
9.4,3.15,5
10.1,3.39,6
12.8,3.71,7
12.8,3.71,7
9.1,3.12,6
12.9,3.47,6
11.5,3.44,7
10.6,3.26,7
11.7,3.26,7
12.8,3.36,7
10.3,3.25,5
12.8,3.36,7
11.7,3.26,7
11.7,3.26,7
12,3.23,7
10.9,3.18,5
12.3,3.25,7
10.4,3.17,6
10,3.23,5
10,3.38,6
11.2,3.22,6
11.4,3.06,6
12.6,3.27,7
12.7,2.89,6
12.7,2.89,6
10.4,3.53,5
11.9,3.2,6
11.9,3.2,6
10.5,3.35,5
12.3,3.27,6
10.5,3.32,7
10.4,3.33,6
12.6,3.41,6
11.6,3.49,5
10.1,3.37,6
10.5,3.32,7
11.5,3.42,7
11,3.35,7
9.6,3.36,5
10.4,3.3,6
9.7,3.2,6
10.6,3.17,7
12.5,3.22,7
9.2,3.35,5
12.6,3.24,7
11.1,3.33,6
10.2,3.58,5
9.8,3.38,6
11.1,3.33,6
10.9,3.29,7
12.2,3.36,6
11.4,3.32,6
10.7,3.31,6
10.4,3.49,5
10.9,3.27,6
10.8,3.27,6
10.4,3.49,5
9.2,3.03,5
12.9,3.56,5
12.7,3.38,7
9.1,3.13,6
9.1,3.13,6
12.1,3.21,7
9.1,3.15,5
11.6,3.19,7
12.1,3.21,7
9.9,3.14,6
12.5,3.21,8
11.4,3.3,6
11.8,3.22,6
11.8,3.28,6
10.2,3.29,6
12.5,3.4,7
10.9,3.02,7
10.9,3.02,7
10.8,3.31,5
11.1,3.18,7
9.2,3.2,5
9.5,3.4,6
10.7,3.31,6
9.2,3.2,5
10.2,3.44,7
11.4,3.15,6
11,3.27,5
11,3.27,5
12.3,3.01,7
11.8,3.16,6
12.3,3.01,7
9.5,3.42,6
12,3.33,6
9.5,3.42,6
9.6,3.12,5
11.6,3.38,7
11.2,3.29,6
9.9,3.02,7
9.9,3.02,7
11.8,2.98,8
11.4,3.35,6
11.5,3.43,6
12,3.33,7
10.8,3.35,6
10.5,3.15,5
10.8,3.35,6
9.4,3.2,5
12.5,3.15,7
9.4,3.2,5
12.4,3.11,6
12,3.35,6
11.2,3.45,6
12,3.35,6
12.4,3.28,6
12.8,3.41,5
12.3,3.21,6
11.7,3.27,7
9.5,3.36,5
10.8,3.17,6
10,3.25,6
12.3,3.78,7
11,3.29,6
9.5,3.12,6
13.6,3.49,6
11.3,3.38,6
11.3,3.38,6
11.3,3.38,6
13.3,3.3,6
12.9,3.52,5
13.1,3.2,8
12.3,3.44,6
12.3,3.45,6
11.2,3.12,6
11.5,3.62,4
11.3,3.14,7
13.5,3.39,6
11.5,3.51,6
10,3.22,5
10.5,3.21,6
10.4,3.18,6
9.5,3.32,5
13.6,3.38,7
11.2,3.37,7
11.8,3.3,7
11.9,3.38,6
11.4,3.22,6
11.4,3.22,6
9.1,3.14,5
9.5,3.34,6
9.5,3.41,6
11,3.27,6
11.4,3.3,6
10,3.61,6
10.1,3.43,5
10.4,3.53,6
9.8,3.28,6
11.8,3.21,7
11.6,3.39,6
12,3.23,6
12.8,3.3,7
12.5,3.46,6
9.8,3.15,5
11.1,3.23,6
11.4,3.59,6
9.8,3.15,5
11.8,3.33,7
12.9,3.46,7
10.6,3.42,6
10.8,3.16,5
11.1,3.14,7
10.2,3.21,6
11.8,3.14,7
10,3.29,5
10,3.29,5
9.4,3.01,5
10.2,2.94,5
12.4,3.32,7
12,3.21,6
11.5,3.32,6
10.5,3.2,6
10.8,3.42,6
12.3,3.27,6
9.5,3.46,6
9.5,3.46,6
11.5,3.61,6
11.2,3.58,4
12.7,3.35,7
11.4,3.68,5
11,3.27,6
11,3.27,6
11.4,3.19,5
10.5,3.38,6
9.3,3.29,5
10.9,3.36,5
11.9,3.33,6
12.2,3.52,5
11.9,3.33,6
10.9,3.36,5
9.6,3.3,4
12.5,3.28,6
10.8,3.56,5
12.4,3.52,7
10.8,3.56,5
9.5,3.22,6
9.5,3.52,6
9.5,3.21,6
9.8,3.16,6
10.9,3.15,6
9.5,3.21,6
9.8,3.16,6
11.2,3.27,7
11.7,3.35,8
9.4,3.14,5
11,3.4,7
11,3.4,7
11,3.4,7
10,3.35,5
11,3.4,7
11.2,3.45,7
10.4,3.35,6
9.3,3.32,5
10.4,3.35,6
10.5,3.15,6
10.4,3.1,6
11.3,3.26,6
9.5,3.29,6
12,3.23,6
10.6,3.24,6
10.9,3.2,6
11.5,3.28,6
11.5,3.28,6
9.6,3.26,6
12,3.23,6
10.3,3.07,6
9.4,3.15,5
9.2,3.02,5
9.6,3.27,5
13.6,3.68,7
9.5,3.4,5
11.8,3.37,6
10.8,3.38,5
9.5,3.4,5
9.3,3.23,4
12.8,3.39,6
11.5,3.3,4
9.4,3.26,6
12.8,3.39,6
9.6,3.38,4
11.8,3.45,4
9.3,3.1,5
9.8,3.16,5
12.2,3.22,6
9.5,3.37,5
10.3,3.33,6
10.5,3.49,5
10,3.12,5
10.5,3.49,5
11.1,3.41,6
10.9,3.4,6
10.9,3.4,6
9.8,3.28,5
9.8,3.45,5
9.5,3.43,5
10.9,3.4,5
11,3.47,5
9.2,3.1,5
11,3.32,6
11.3,3.44,6
11.3,3.44,6
9.3,3.02,5
11.2,3.59,4
9.8,3.15,5
9.6,3.37,4
11.8,3.38,6
10.3,3.38,6
10.3,3.38,6
11.4,3.1,6
9.5,3.47,6
14,3.5,8
14,3.7,6
11.7,3.38,6
11.2,3.5,5
9.3,3.1,5
11.2,3.24,6
9.5,3.24,6
12,3.28,4
9.7,3.35,6
9.5,3.24,6
11.5,3.14,7
10.4,3.37,6
10.4,3.37,6
10.9,3.33,6
10.2,3.26,6
10.9,3.39,5
10.5,3.15,5
12.4,3.35,6
13,3.22,5
10.2,3.3,5
10.2,3.3,5
11,3.36,5
10.9,3.29,6
12.4,3.62,6
9.7,3.34,4
10.9,3.29,6
9.5,3.2,5
9.5,3.2,5
12.1,3.35,6
12.2,3.45,6
10.9,3.5,3
12.5,3.78,6
11.7,3.6,6
11.2,3.3,6
11.2,3.36,5
9.2,3.12,5
9.8,3.3,5
10.1,3.29,5
11.1,3.41,4
10.1,3.29,5
9.2,3.28,5
9.8,3.3,5
12.8,3.33,6
10.5,3.34,5
10.1,3.4,6
10.1,3.4,6
9.8,3.26,6
12.5,4.01,6
11.9,3.18,6
9.8,3.26,6
9.1,2.9,6
10.1,3.26,5
12.5,4.01,6
11.4,3.18,5
11,3.18,7
10.6,3.39,6
10.6,3.39,6
10.6,3.39,6
10.6,3.39,6
9.3,3.34,5
9.8,3.29,6
9.8,3.29,6
9.2,3.26,5
9.7,3.09,6
9.6,3.18,5
10,3.39,5
12.6,3.38,6
9.5,3.36,5
9.5,3.36,5
9.5,3.36,5
10.5,3.36,6
10.5,3.36,6
10.5,3.36,6
10.4,3.37,6
10.5,3.36,6
11,3.09,5
10.2,3.25,6
11.4,3.52,5
9.5,3.34,5
9.5,3.34,5
10.8,3.41,5
10,3.34,5
11.8,3.32,6
10.3,3.37,5
10.3,3.37,5
10,3.35,5
10.1,3.36,5
10.1,3.36,5
11.1,3.6,6
9.5,3.28,5
10.0333333333333,3.07,6
10.9,3.24,5
10.1,3.36,5
10.0333333333333,3.07,6
9.8,3.29,4
11.8,3.46,6
9.5,3.18,5
9.8,3.24,5
10.5,3.25,6
9.8,3.27,6
10.4,3.33,4
9.2,3,5
11.6,3.21,6
9.2,3,5
9.3,3.24,5
9.8,3.32,3
9.9,3.26,5
10,3.11,5
12.2,3.71,6
9.9,3.29,6
10.8,3.36,6
10.8,3.36,6
9.6,3.38,5
9.9,3.3,5
9.9,3.3,5
9.5,3.19,5
9.2,3.19,5
9.6,3.3,5
9.6,3.3,5
10.3,3.35,5
9.7,3.1,5
12.5,3.58,6
11,3.37,5
9.8,3.17,5
10.2,3.3,5
9.2,3.32,5
9.55,3.31,6
9.55,3.31,5
9.9,3.28,5
10.2,3.52,7
11.1,3.42,6
9.9,3.25,5
9.9,3.25,5
11.9,3.44,6
10,3.23,8
10.7,3.43,6
11.3,3.28,7
10.9,3.22,6
10.7,3.54,6
12.4,3.27,7
10.7,3.54,6
11.9,3.5,6
10.6,3.56,6
10.9,3.22,6
10.1,3.04,5
9.6,3.2,5
9.4,3.56,5
9.6,3.2,5
12.1,3.35,7
9.8,3.16,5
9.9,3.27,5
9.8,3.16,5
9.4,3.34,5
11.4,3.39,6
11,3.5,4
9.6,3.26,6
9.6,3.26,6
12.1,3.41,6
11.2,3.3,5
11,3.31,5
11.3,3.33,5
10.4,3.37,5
9.9,3.26,6
11.7,3.48,6
10.1,3.42,7
9,3.18,6
9,3.18,6
8.5,3.15,5
11.1,3.53,5
10.3,3.41,5
11.0666666666667,3.32,6
11.3,3.23,7
9.56666666666667,3.16,6
9.8,3.4,5
11.7,3.38,5
11.1,3.32,6
9.6,3.16,6
9.8,3.4,5
9.7,3.4,5
9.7,3.4,5
11.3,3.23,8
11.3,3.23,7
11,3.33,7
11.6,3.35,7
9,3.41,5
10.9,3.21,6
10.9,3.5,6
10.55,3.3,6
9,3.41,5
10.5,3.39,5
11.9,3.32,7
10.4,3.41,6
10,3.59,4
10.4,3.36,6
10.8,3.34,6
9.7,3.41,5
9.7,3.41,5
10,3.3,7
11,3.52,4
10,3.3,7
9.7,3.31,3
9.7,2.88,5
12.6,3.43,5
11.1,3.38,6
11,3.38,5
8.8,3.16,5
13.5666666666667,3.54,7
8.8,3.16,5
13.6,3.54,7
10.2,3.4,3
10.6,3.39,5
10.1,3.66,4
10.6,3.39,5
10.3,3.33,4
10.7,3.35,5
10.9,3.53,4
9.7,3.37,5
10.3,3.38,5
10.6,3.52,5
11.4,3.72,5
10.8,3.44,6
12.4,3.39,6
11.4,3.72,5
11.95,3.57,5
9.7,3.26,5
10,3.42,7
10.6,3.35,6
9.7,3.26,5
11.5,3.45,6
10.8,3.53,6
11.5,3.45,6
9.6,3.26,5
9.4,3.16,5
9.8,3.31,5
11,3.51,6
11.4,3.36,6
9.95,3.55,3
10.8,3.51,6
11.4,3.36,6
10.6,3.38,6
11.1,3.28,5
9.7,3.47,6
9.8,3.44,5
10.4,3.53,6
10.5,3.33,6
9.23333333333333,3.53,6
9.25,3.53,6
10.6,3.42,5
10.3,3.38,6
10.5,3.32,5
10.2,3.44,5
10.3,3.38,6
9.05,3.39,4
10.6,3.42,5
10.4,3.34,5
10,3.39,6
9.7,3.33,5
9.6,3.3,6
10.8,3.29,6
10.1,3.31,6
10.2,3.42,6
11.3,3.42,6
10.2,3.6,5
9.9,3.21,6
9,3.39,5
11.7,3.42,7
9.7,3.36,6
10.3,3.6,6
10.2,3.5,6
12,3.57,5
9.9,3.46,5
11.6,3.56,6
10.9,3.4,7
10.75,3.3,6
10.4,3.25,6
11.2,3.17,7
10.1,3.36,6
10.2,3.36,5
12.1,3.55,5
10.4,3.1,5
11.4,3.24,8
9.5,3.45,5
9.4,3.44,5
11.3,3.51,6
9.4,3.41,5
11,3.54,6
10.55,3.44,7
9.9,3.53,5
11,3.54,6
9.5,3.22,5
9.9,3.21,5
9.9,3.21,5
9.9,3.21,5
10.1,3.29,5
10.1,3.29,5
10.1,3.29,5
10.9,3.39,6
11.2,3.34,6
10.1,3.29,5
9.2,3.34,5
11.5,3.48,6
12.4,3.37,6
11.1,3.44,6
9.5,3.33,5
12.5,3.58,6
10.5,3.39,6
11.8,3.26,6
10.8,3.3,6
11.9,3.54,6
11.3,3.42,6
11.3,3.54,5
11.9,3.36,6
11.3,3.54,5
11.9,3.57,5
9.8,3.33,5
11.6,3.29,7
11.5,3.3,6
11.4,3.34,6
10.9,3.55,6
12.8,3.27,6
9.2,3.29,5
11.6,3.32,6
11.6,3.67,6
11,3.42,6
9.5,3.42,6
10.5,3.45,5
11.2,3.52,6
11,3.42,6
10.2,3.57,5
11,3.39,6
'''

In [4]:
from tempfile import mkdtemp

workspace = Path(mkdtemp(prefix="gortsup_files_", dir=Path.cwd()))
wine_path = workspace / "wine_measurements.csv"
with open(wine_path, "w", encoding="utf-8", newline="") as file:
    file.write(DATA_CSV)

print("Դասի աշխատանքային պանակը", workspace)
print("Իրական տվյալների ֆայլը", wine_path)


Դասի աշխատանքային պանակը /home/developer/Documents/MyPythonCodes/GortsUp/Working_with_Files/gortsup_files_hnafe5q4
Իրական տվյալների ֆայլը /home/developer/Documents/MyPythonCodes/GortsUp/Working_with_Files/gortsup_files_hnafe5q4/wine_measurements.csv


Նախապատրաստման բջիջը յուրաքանչյուր գործարկմամբ ստեղծում է նոր պանակ՝ notebook-ի աշխատանքային պանակում։ Այնտեղ կարող եք ինքնուրույն ստեղծել ձեր վերլուծության արդյունքները։ Assert ստուգումների ելքային ֆայլերը ստեղծվում և մաքրվում են առանձին ժամանակավոր պանակներում։

## Տեսություն · ուղիներ և ֆայլի բացման ռեժիմներ
Հարաբերական ուղին հաշվվում է աշխատանքային պանակից՝ `Path.cwd()`-ից։ Բացարձակ ուղին ամբողջությամբ նշում է տեղադրությունը։ Ուղիները միացրեք `Path / "name"` ձևով։ `resolve()`-ը ստանում է բացարձակ ձևը, իսկ `mkdir(parents=True, exist_ok=True)`-ը ստեղծում է նաև անհրաժեշտ ծնող պանակները։

| Ռեժիմ | Գործողություն |
|---|---|
| r | Կարդալ գոյություն ունեցող ֆայլը |
| w | Ստեղծել կամ դատարկել և գրել |
| a | Ավելացնել վերջում |
| x | Ստեղծել միայն նոր ֆայլ, եղածի դեպքում բարձրացնել FileExistsError |
| rb / wb | Կարդալ կամ գրել բայթեր |

Տեքստի համար հստակ նշեք `encoding="utf-8"`։ `with`-ը փակում է ֆայլը նաև բլոկում սխալ առաջանալու դեպքում։ `read()`-ը կարող է ամբողջ ֆայլը բերել հիշողության մեջ, իսկ `for line in file`-ը թույլ է տալիս հերթով մշակել տողերը։ Մեկ հսկայական տողը նույնպես կարող է շատ հիշողություն պահանջել։

## Վարժություն 1 · Նախագծի պանակներ

Գրեք `make_project(root)`։ Ստեղծեք `raw`, `processed`, `reports` ենթապանակները։ Վերադարձրեք անունից դեպի Path բառարան։ Կրկնակի կանչը պետք է աշխատի։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [5]:
def make_project(root):
    paths = {}
    for name in ("raw", "processed", "reports"):
        path = Path(root) / name
        path.mkdir(parents=True, exist_ok=True)
        paths[name] = path
    return paths

### Assert ստուգումներ

In [12]:
def check_01():
    with TemporaryDirectory(dir = workspace, delete = False) as folder:
        root = Path(folder)
        paths = make_project(root / "project")
        assert set(paths) == {"raw", "processed", "reports"}
        assert all(path.is_dir() for path in paths.values())
        assert make_project(root / "project") == paths

check_01()
print("Վարժություն 1․ ստուգումներն անցան։")

Վարժություն 1․ ստուգումներն անցան։


## Վարժություն 2 · Նոր ֆայլի պաշտպանված ստեղծում

Գրեք `create_note(path, text)`՝ x ռեժիմով և UTF-8 կոդավորմամբ։ Գոյություն ունեցող ֆայլի դեպքում թողեք `FileExistsError`-ը հասնի կանչողին։ Ծնող պանակը արդեն կա։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [7]:
def create_note(path, text):
    with open(path, "x", encoding="utf-8") as file:
        file.write(text)

### Assert ստուգումներ

In [8]:
def check_02():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "note.txt"
        create_note(path, "Հայերեն")
        assert path.read_text(encoding="utf-8") == "Հայերեն"
        try:
            create_note(path, "Նոր")
        except FileExistsError:
            pass
        else:
            assert False
        assert path.read_text(encoding="utf-8") == "Հայերեն"

check_02()
print("Վարժություն 2․ ստուգումներն անցան։")

Վարժություն 2․ ստուգումներն անցան։


## Վարժություն 3 · Գործողությունների մատյան

Գրեք `append_log(path, message)`։ Վերջի նոր տողի նշանները հեռացրեք և ավելացրեք մեկ նոր տողի նշանով։ Օգտագործեք a ռեժիմ։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def append_log(path, message):
    with open(path, "a", encoding="utf-8") as file:
        file.write(message.rstrip("\r\n") + "\n")

### Assert ստուգումներ

In [ ]:
def check_03():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "actions.txt"
        append_log(path, "Սկիզբ")
        append_log(path, "Ավարտ\n")
        assert path.read_text(encoding="utf-8").splitlines() == ["Սկիզբ", "Ավարտ"]

check_03()
print("Վարժություն 3․ ստուգումներն անցան։")

## Վարժություն 4 · Մեծ տեքստի տողային մշակում

Գրեք `count_nonempty_lines(path)`։ for ցիկլով հաշվեք ոչ դատարկ տողերը։ Միայն բացատներ պարունակող տողը դատարկ համարեք։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def count_nonempty_lines(path):
    count = 0
    with open(path, encoding="utf-8") as file:
        for line in file:
            if line.strip():
                count += 1
    return count

### Assert ստուգումներ

In [ ]:
def check_04():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "input.txt"
        path.write_text("Անի\n\n   \nՄարի\n", encoding="utf-8")
        assert count_nonempty_lines(path) == 2
        path.write_text("", encoding="utf-8")
        assert count_nonempty_lines(path) == 0

check_04()
print("Վարժություն 4․ ստուգումներն անցան։")

## Տեսություն · CSV կառուցվածք և կոդավորում
CSV-ի մեջ կարող են լինել ստորակետեր և նույնիսկ նոր տողեր պարունակող չակերտված դաշտեր։ Օգտագործեք `csv` մոդուլը։ Ֆայլը բացեք `newline=""`-ով, որպեսզի մոդուլը կառավարի տողերի ավարտները։

`DictReader`-ը դաշտերը վերադարձնում է որպես տողեր։ Թվերը փոխարկեք առանձին քայլում։ `delimiter=";"`-ը նշում է կետ-ստորակետ բաժանարար։ `utf-8-sig`-ը ընթերցելիս հեռացնում է սկզբի UTF-8 BOM-ը, եթե այն կա։ Սխալ կոդավորումը կարող է առաջացնել UnicodeDecodeError։

Դատարկ բովանդակությունը, պակասող սյունակները և սխալ թիվը տարբեր խնդիրներ են։ Ստուգեք ֆայլի կառուցվածքը մինչև չափումների հաշվարկը։

## Վարժություն 5 · Իրական CSV-ի ընթերցում

Գրեք `read_measurements(path)`։ Օգտագործեք DictReader, newline="" և utf-8-sig։ Պահանջեք alcohol, pH, quality սյունակները։ Բացակայող վերնագրերի դեպքում բարձրացրեք ValueError։ Վերադարձրեք տողերի ցուցակը։ Միայն վերնագրերով CSV-ի դեպքում վերադարձեք []։ Աշխատեք ներքևում տրված իրական wine_path ֆայլի հետ։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
FIELDS = ["alcohol", "pH", "quality"]

def read_measurements(path):
    with open(path, encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        if not set(FIELDS).issubset(reader.fieldnames or []):
            raise ValueError("Պարտադիր սյունակները բացակայում են։")
        return list(reader)

### Assert ստուգումներ

In [ ]:
def check_05():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        rows = read_measurements(wine_path)
        assert len(rows) == 1599
        assert rows[0] == {"alcohol": "9.4", "pH": "3.51", "quality": "5"}
        path = root / "empty.csv"
        path.write_text("alcohol,pH,quality\n", encoding="utf-8-sig")
        assert read_measurements(path) == []
        path.write_text("alcohol,quality\n", encoding="utf-8")
        try:
            read_measurements(path)
        except ValueError:
            pass
        else:
            assert False

check_05()
print("Վարժություն 5․ ստուգումներն անցան։")

## Վարժություն 6 · Չափումների վավերացում

Գրեք `parse_measurement(row)`։ alcohol-ը և pH-ը դարձրեք float, quality-ն՝ int։ Պահանջեք վերջավոր թվեր, alcohol ≥ 0, 0 ≤ pH ≤ 14 և 0 ≤ quality ≤ 10։ Սխալի դեպքում բարձրացրեք ValueError։ Վերադարձրեք նոր բառարան։ Սյունակներն առկա են և արժեքները տողային են։ Սրանք դասի վավերացման կանոններն են։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def parse_measurement(row):
    alcohol = float(row["alcohol"])
    ph = float(row["pH"])
    quality = int(row["quality"])
    if not isfinite(alcohol) or not isfinite(ph):
        raise ValueError("Չափումը պետք է վերջավոր լինի։")
    if alcohol < 0 or not 0 <= ph <= 14 or not 0 <= quality <= 10:
        raise ValueError("Արժեքը թույլատրելի միջակայքից դուրս է։")
    return {"alcohol": alcohol, "pH": ph, "quality": quality}

### Assert ստուգումներ

In [ ]:
def check_06():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        row = read_measurements(wine_path)[0]
        assert parse_measurement(row) == {"alcohol": 9.4, "pH": 3.51, "quality": 5}
        assert row["quality"] == "5"
        for field, value in [("alcohol", "nan"), ("pH", "15"), ("quality", "11"), ("alcohol", "?")]:
            try:
                parse_measurement({**row, field: value})
            except ValueError:
                pass
            else:
                assert False

check_06()
print("Վարժություն 6․ ստուգումներն անցան։")

## Վարժություն 7 · Վավեր և մերժված չափումներ

Գրեք `clean_measurements(rows)`։ Յուրաքանչյուր գրառումը մշակեք parse_measurement()-ով։ ValueError-ի դեպքում պահեք record_number, row, reason դաշտերով գրառում։ Համարակալումը սկսեք 1-ից։ Վերադարձրեք clean, rejected զույգը։ Իրական տվյալները ստուգելուց հետո փորձեք նաև թեստերում դիտավորյալ վնասված գրառումը։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def clean_measurements(rows):
    clean, rejected = [], []
    for number, row in enumerate(rows, start=1):
        try:
            clean.append(parse_measurement(row))
        except ValueError as error:
            rejected.append({"record_number": number, "row": row, "reason": str(error)})
    return clean, rejected

### Assert ստուգումներ

In [ ]:
def check_07():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        rows = read_measurements(wine_path)
        clean, rejected = clean_measurements(rows)
        assert len(clean) == 1599 and rejected == []
        bad_row = {"alcohol": "?", "pH": "3.0", "quality": "5"}
        clean, rejected = clean_measurements([rows[0], bad_row])
        assert len(clean) == len(rejected) == 1
        assert rejected[0]["record_number"] == 2
        assert rejected[0]["reason"]
        assert clean_measurements([]) == ([], [])

check_07()
print("Վարժություն 7․ ստուգումներն անցան։")

## Վարժություն 8 · Մաքրված CSV արտահանում

Գրեք `write_clean_csv(path, rows)`։ DictWriter-ով պահպանեք alcohol, pH, quality հերթականությամբ։ Դատարկ ցուցակի դեպքում էլ վերնագրերը գրեք։ Վերաբացեք արդյունքը և ստուգեք թվային արժեքների պահպանումը։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def write_clean_csv(path, rows):
    with open(path, "w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rows)

### Assert ստուգումներ

In [ ]:
def check_08():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        clean, _ = clean_measurements(read_measurements(wine_path))
        path = root / "clean.csv"
        write_clean_csv(path, clean)
        restored = [parse_measurement(row) for row in read_measurements(path)]
        assert restored == clean
        write_clean_csv(path, [])
        assert read_measurements(path) == []

check_08()
print("Վարժություն 8․ ստուգումներն անցան։")

## Տեսություն · JSON JSONL և տվյալների վերականգնում
JSON-ը պահում է մեկ կառուցվածք։ JSONL ձևաչափում յուրաքանչյուր տողը առանձին JSON արժեք է։ Այն հարմար է հերթով ավելացվող կամ հերթով մշակվող գրառումների համար։ JSON ֆայլի վերջում երկրորդ առանձին object ավելացնելը այն JSONL չի դարձնում, եթե յուրաքանչյուր գրառում առանձին տողում չէ։

`ensure_ascii=False`-ը պահում է հայերեն ընթեռնելի տեսքը։ `allow_nan=False`-ը արգելում է NaN և Infinity արժեքների արտահանումը։ Բացակայող ֆայլը և վնասված JSON-ը տարբեր կերպ մշակեք։ Վնասված ֆայլը լուռ դատարկ բառարանով փոխարինելը կարող է ծածկել խնդիրը։

## Վարժություն 9 · JSON ամփոփում

Գրեք `save_summary(path, summary)`՝ UTF-8, ընթեռնելի հայերեն, երկու բացատով նահանջ և allow_nan=False կարգավորումներով։ Այս պարզ տարբերակը վերագրում է ֆայլը։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def save_summary(path, summary):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(summary, file, ensure_ascii=False, indent=2, allow_nan=False)

### Assert ստուգումներ

In [ ]:
def check_09():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "summary.json"
        data = {"ապրանք": "Թեյ", "count": 2}
        save_summary(path, data)
        assert json.loads(path.read_text(encoding="utf-8")) == data
        assert "Թեյ" in path.read_text(encoding="utf-8")

check_09()
print("Վարժություն 9․ ստուգումներն անցան։")

## Վարժություն 10 · JSON բեռնում և սխալների տարբերակում

Գրեք `load_summary(path)`։ Բացակայող ֆայլի դեպքում վերադարձեք None։ Վնասված JSON-ի դեպքում թողեք JSONDecodeError-ը փոխանցվի։ Պահանջեք վերին մակարդակում dict, հակառակ դեպքում բարձրացրեք ValueError։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def load_summary(path):
    try:
        with open(path, encoding="utf-8") as file:
            data = json.load(file)
    except FileNotFoundError:
        return None
    if not isinstance(data, dict):
        raise ValueError("Ամփոփումը պետք է լինի dictionary։")
    return data

### Assert ստուգումներ

In [ ]:
def check_10():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "summary.json"
        assert load_summary(path) is None
        path.write_text('{"count":2}', encoding="utf-8")
        assert load_summary(path) == {"count": 2}
        path.write_text("{broken", encoding="utf-8")
        try:
            load_summary(path)
        except json.JSONDecodeError:
            pass
        else:
            assert False
        assert path.read_text(encoding="utf-8") == "{broken"

check_10()
print("Վարժություն 10․ ստուգումներն անցան։")

## Վարժություն 11 · Մերժված տողերը JSONL ձևաչափով

Գրեք `write_jsonl(path, records)`։ Յուրաքանչյուր գրառումը գրեք մեկ JSON տողով և ավարտեք նոր տողի նշանով։ Օգտագործեք ensure_ascii=False, allow_nan=False։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")

### Assert ստուգումներ

In [ ]:
def check_11():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "rejected.jsonl"
        data = [{"reason":"Սխալ թիվ"}, {"reason":"Դատարկ ապրանք"}]
        write_jsonl(path, data)
        lines = path.read_text(encoding="utf-8").splitlines()
        assert len(lines) == 2
        assert [json.loads(line) for line in lines] == data
        write_jsonl(path, [])
        assert path.read_text() == ""

check_11()
print("Վարժություն 11․ ստուգումներն անցան։")

## Վարժություն 12 · JSONL տողային հաշվարկ

Գրեք `count_jsonl(path)`։ Հաշվեք ոչ դատարկ տողերը՝ յուրաքանչյուրն անցկացնելով json.loads()-ով։ Վնասված տողի դեպքում թողեք JSONDecodeError-ը հասնի կանչողին։ Օգտագործեք for line in file։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [13]:
def count_jsonl(path):
    count = 0
    with open(path, encoding="utf-8") as file:
        for line in file:
            if line.strip():
                json.loads(line)
                count += 1
    return count

### Assert ստուգումներ

In [14]:
def check_12():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "data.jsonl"
        path.write_text('{"x":1}\n\n{"x":2}\n', encoding="utf-8")
        assert count_jsonl(path) == 2
        path.write_text('{"x":1}\nbroken\n', encoding="utf-8")
        try:
            count_jsonl(path)
        except json.JSONDecodeError:
            pass
        else:
            assert False

check_12()
print("Վարժություն 12․ ստուգումներն անցան։")

Վարժություն 12․ ստուգումներն անցան։


## Տեսություն · ֆայլի հայտնաբերում և անվտանգ փոխարինում
`glob("*.csv")`-ը գտնում է համապատասխան ֆայլերը մեկ պանակում, `rglob("*.csv")`-ը՝ նաև ենթապանակներում։ Արդյունքը դասավորեք, եթե հերթականությունը կարևոր է։ `stat().st_size`-ը վերադարձնում է բայթերի քանակը։ Տեքստի նիշերի քանակը կարող է տարբերվել UTF-8 բայթերի քանակից։

`w` ռեժիմը բացելու պահին դատարկում է եղած ֆայլը։ Ավելացնենք պահպանում ժամանակավոր ֆայլում և նույն պանակում `os.replace()`-ով փոխարինում։ Սա պաշտպանում է հին ֆայլը JSON-ի սերիալացման ձախողման դեպքում։ Այս մոտեցումը չի լուծում միաժամանակ գրող ծրագրերի կոնֆլիկտը և ինքնին էլեկտրամատակարարման կորստից պաշտպանության երաշխիք չէ։

## Վարժություն 13 · Մուտքային CSV ֆայլերի որոնում

Գրեք `list_csv_files(folder)`։ Վերադարձրեք միայն տվյալ պանակի .csv վերջավորությամբ սովորական ֆայլերը՝ անունով դասավորված։ Ենթապանակների մեջ մի մտեք։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [15]:
def list_csv_files(folder):
    return sorted((path for path in Path(folder).glob("*.csv") if path.is_file()),
                  key=lambda path: path.name)

### Assert ստուգումներ

In [16]:
def check_13():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        (root / "b.csv").write_text("")
        (root / "a.csv").write_text("")
        (root / "note.txt").write_text("")
        (root / "fake.csv").mkdir()
        assert [p.name for p in list_csv_files(root)] == ["a.csv", "b.csv"]

check_13()
print("Վարժություն 13․ ստուգումներն անցան։")

Վարժություն 13․ ստուգումներն անցան։


## Վարժություն 14 · Անվտանգ JSON պահպանում

Գրեք `safe_save_json(path, data)`։ Ծնող պանակն արդեն կա։ Ստեղծեք ժամանակավոր UTF-8 ֆայլ նույն պանակում, գրեք strict JSON, փակեք այն և os.replace()-ով փոխարինեք նպատակային ֆայլը։ Վերջում մաքրեք մնացած ժամանակավոր ֆայլը նաև սխալի դեպքում։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def safe_save_json(path, data):
    path = Path(path)
    temporary = None
    try:
        with NamedTemporaryFile(mode="w", encoding="utf-8",
                                dir=path.parent, suffix=".tmp", delete=False) as file:
            temporary = Path(file.name)
            json.dump(data, file, ensure_ascii=False, allow_nan=False, indent=2)
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)

### Assert ստուգումներ

In [ ]:
def check_14():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        path = root / "summary.json"
        safe_save_json(path, {"total": 7})
        assert load_summary(path) == {"total": 7}
        before = path.read_bytes()
        try:
            safe_save_json(path, {"total": float("nan")})
        except ValueError:
            pass
        else:
            assert False
        assert path.read_bytes() == before
        assert list(root.glob("*.tmp")) == []

check_14()
print("Վարժություն 14․ ստուգումներն անցան։")

## Վարժություն 15 · Իրական տվյալներից վերլուծական հաշվետվություն

Գրեք `run_pipeline(input_path, output_root)`։ Կարդացեք CSV-ն, վավերացրեք, գրեք processed/clean.csv և reports/rejected.jsonl։ reports/summary.json-ում safe_save_json()-ով գրեք total, accepted, rejected, mean_alcohol, high_quality_count։ Վերջին դաշտը quality ≥ 7 չափումների թիվն է։ Միջին ալկոհոլը կլորացրեք երկու նիշով, իսկ վավեր գրառումների բացակայության դեպքում պահեք None։ Վերադարձրեք ամփոփումը։ Օգտագործեք նախկին ֆունկցիաները։ Կրկնվող չափումները պահեք՝ առանց դրանք ավտոմատ հեռացնելու։

In [ ]:
# Գրեք ձեր լուծումն այստեղ։

### Առաջարկվող լուծում

In [ ]:
def run_pipeline(input_path, output_root):
    paths = make_project(output_root)
    rows = read_measurements(input_path)
    clean, rejected = clean_measurements(rows)
    mean_alcohol = None
    if clean:
        mean_alcohol = round(sum(row["alcohol"] for row in clean) / len(clean), 2)
    summary = {
        "total": len(rows),
        "accepted": len(clean),
        "rejected": len(rejected),
        "mean_alcohol": mean_alcohol,
        "high_quality_count": sum(row["quality"] >= 7 for row in clean),
    }
    write_clean_csv(paths["processed"] / "clean.csv", clean)
    write_jsonl(paths["reports"] / "rejected.jsonl", rejected)
    safe_save_json(paths["reports"] / "summary.json", summary)
    return summary

### Assert ստուգումներ

In [ ]:
def check_15():
    with TemporaryDirectory() as folder:
        root = Path(folder)
        before = wine_path.read_bytes()
        summary = run_pipeline(wine_path, root / "analysis")
        assert summary == {"total": 1599, "accepted": 1599, "rejected": 0,
                           "mean_alcohol": 10.42, "high_quality_count": 217}
        assert summary["total"] == summary["accepted"] + summary["rejected"]
        assert load_summary(root / "analysis" / "reports" / "summary.json") == summary
        assert count_jsonl(root / "analysis" / "reports" / "rejected.jsonl") == 0
        assert wine_path.read_bytes() == before
        assert run_pipeline(wine_path, root / "analysis") == summary
        empty = root / "empty.csv"
        empty.write_text("alcohol,pH,quality\n", encoding="utf-8")
        assert run_pipeline(empty, root / "empty_output")["mean_alcohol"] is None

check_15()
print("Վարժություն 15․ ստուգումներն անցան։")

## Արդյունքի քննարկում
Իրական տվյալներում 1,599 չափումներից 217-ը ունեն առնվազն 7 գնահատական։ Միջին ալկոհոլային պարունակությունը մոտ 10.42% է։ Ձեր հաշվետվությունը պետք է հաստատի այս արժեքները։

Քննարկեք, թե որ ձևաչափն է հարմար աղյուսակի, մերժման պատճառների և ներդրված ամփոփման համար։ Օրինակ ցույց տվեք, թե ինչու ֆիզիկական տողը միշտ նույնը չէ, ինչ CSV գրառումը։

Անվտանգ JSON փոխարինումը պաշտպանում է մեկ ֆայլը։ Ամբողջ շղթայի երեք ելքերը պահպանվում են հաջորդաբար։ Միաժամանակյա գրառումների կամ մասնակի ձախողման դեպքում ընդհանուր հետևողականությունը պահանջում է հավելյալ նախագծում։

**Հանձնեք** ձեր լրացված notebook-ը, clean.csv-ը և summary.json-ը։ Կարճ մեկնաբանեք, որ այստեղ մերժված տողերի բացակայությունը վերաբերում է սահմանված ստուգումներին։
